In [49]:
using LowLevelFEM
using LinearAlgebra
using SparseArrays
using Test

spectral_path = joinpath(
    dirname(pathof(LowLevelFEM)),
    "spectral.jl"
)

@assert isfile(spectral_path) "Copy spectral.jl into the LowLevelFEM src directory first: $spectral_path"

Base.include(
    LowLevelFEM,
    spectral_path
)

using LowLevelFEM: spectralTransformation

┌ Warning: Replacing docs for `LowLevelFEM._sem_legendre_pair :: Tuple{Integer, Real}` in module `LowLevelFEM`
└ @ Base.Docs docs/Docs.jl:253
┌ Warning: Replacing docs for `LowLevelFEM._sem_gll :: Tuple{Integer}` in module `LowLevelFEM`
└ @ Base.Docs docs/Docs.jl:253
┌ Warning: Replacing docs for `LowLevelFEM._sem_barycentric_weights :: Tuple{AbstractVector}` in module `LowLevelFEM`
└ @ Base.Docs docs/Docs.jl:253
┌ Warning: Replacing docs for `LowLevelFEM._sem_lagrange_matrix :: Tuple{AbstractVector, AbstractVector}` in module `LowLevelFEM`
└ @ Base.Docs docs/Docs.jl:253
┌ Warning: Replacing docs for `LowLevelFEM._sem_insert_unique! :: Tuple{Dict{Tuple{Int64, Int64}, Float64}, Integer, Integer, Real}` in module `LowLevelFEM`
└ @ Base.Docs docs/Docs.jl:253
┌ Warning: Replacing docs for `LowLevelFEM._sem_sparse_from_dict :: Tuple{Dict{Tuple{Int64, Int64}, Float64}, Integer, Integer}` in module `LowLevelFEM`
└ @ Base.Docs docs/Docs.jl:253
┌ Warning: Replacing docs for `LowLevelFEM._sem_ch

In [50]:
structured_rect_mesh(n=10, order=4)

mat = Material("body")

P = Problem(
    [mat];
    type=:VectorField,
    dim=2,
    fieldName=:u,
    basis=:spectral
)

support = BoundaryCondition("left", ux=0.1, uy=0)
K = ∫(
    Grad(P) ⋅ Grad(P);
    Ω="body"
)

f = ∫(
    P ⋅ [1,0];
    Ω="body"
)

@time u = solveField(
    Symmetric(K),
    f;
    support=[support]
);

  2.522595 seconds (2.16 M allocations: 153.999 MiB, 4.60% gc time, 89.32% compilation time: 98% of which was recompilation)


In [51]:
showDoFResults(u, name="u spectral")

0

In [52]:
#line_mesh(n=3, order=3)

#mat = Material("body")

P2 = Problem(
    [mat];
    type=:VectorField,
    dim=2,
    fieldName=:u,
    basis=:lagrange
)

support = BoundaryCondition("left", ux=0.1, uy=0)
K = ∫(
    Grad(P2) ⋅ Grad(P2);
    Ω="body"
)

f = ∫(
    P2 ⋅ [1,0];
    Ω="body"
)

@time u2 = solveField(
    Symmetric(K),
    f;
    support=[support]
);

  0.012567 seconds (4.14 k allocations: 17.551 MiB, 9.54% gc time)


In [53]:
showDoFResults(u2, name="u lagrange")

1

In [54]:
norm(DoFs(u) - DoFs(u2)) / norm(DoFs(u))

2.0166444703362273e-13

In [55]:
Ts, Rs = LowLevelFEM.field_transformation(P)
Tl, Rl = LowLevelFEM.field_transformation(P2)

@show P.basis
@show P2.basis

@show size(Ts)
@show size(Tl)

@show nnz(Ts)
@show nnz(Tl)

@show norm(Ts - Tl, Inf)

P.basis = :spectral
P2.basis = :lagrange
size(Ts) = (3362, 3362)
size(Tl) = (3362, 3362)
nnz(Ts) = 29282
nnz(Tl) = 3362
norm(Ts - Tl, Inf) = 1.1171875


1.1171875

In [56]:
Tl

3362×3362 SparseMatrixCSC{Float64, Int64} with 3362 stored entries:
⎡⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⎥
⎣⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⎦

In [57]:
Ts

3362×3362 SparseMatrixCSC{Float64, Int64} with 29282 stored entries:
⎡⠃⠀⠀⠀⠈⠃⠀⠀⠘⠃⠀⠀⠘⠀⠀⠀⠘⠀⠀⠀⠺⠀⠀⠀⠇⠀⠀⠀⠆⠀⠀⠀⠆⠀⠀⢨⣤⣤⣤⠌⎤
⎢⣠⣤⣤⣤⡤⠇⠀⠀⠸⠀⠀⠀⠘⠀⠀⠀⠘⠀⠀⠀⠃⠀⠀⠀⠃⠀⠀⠀⠃⠀⠀⠐⠃⠀⠀⠘⠀⠀⠈⠉⎥
⎢⣈⠉⠉⠉⠉⠈⠉⠉⠉⠉⠓⠒⠂⠒⠒⠒⠂⠒⠒⠦⠤⠤⠤⠤⠠⠤⠤⠤⢠⣀⣀⣀⣀⣀⣀⣀⠀⠀⠀⠀⎥
⎢⠙⠳⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣠⡀⠈⠙⠷⣤⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠙⠷⣄⡀⠉⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠈⠛⢶⣀⠈⠻⢦⣀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠙⢧⣄⠀⠛⢶⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⣤⠀⠙⢳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣆⡀⠉⠷⣤⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠹⢦⡀⠈⠻⢦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢶⣄⠈⠛⢦⣀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⢳⣄⠀⠙⢷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⣤⡀⠙⠳⣦⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠛⢦⡀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢶⣀⠈⠹⢧⣀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢷⣄⡀⠛⢶⣄⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⣤⠀⠙⢳⣆⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠛⢦⣀⠈⠻⣤⡀⠀⠀⎥
⎣⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠳⣦⠀⠙⠳⣄⎦

In [58]:
#openPostProcessor()

---

In [59]:
P = Problem(
    [mat];
    type=:ScalarField,
    dim=2,
    field=:u,
    basis=:spectral
)

Problem("structured_rect", :ScalarField, 2, 1, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 1681, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :rhs, false, :spectral)

In [60]:
sem =
    LowLevelFEM.spectralTransformation(
        P;
        details=true,
        check=true
    )

@show size(sem.T)
@show size(sem.R)
@show sem.inverse_error
@show sem.spectral_coordinates
@show sem.spectral_connectivity

size(sem.T) = (1681, 1681)
size(sem.R) = (1681, 1681)
sem.inverse_error = 4.884981308350689e-15
sem.spectral_coordinates = [-3.198830089701234e-16 -1.5681900222830298e-16 0.0; 0.017267316464601192 -1.0096090630185018e-16 0.0; 0.05000000000000001 -1.3106799596269226e-17 0.0; 0.08273268353539869 -2.321060010856968e-16 0.0; 0.09999999999999974 1.498801083243962e-16 0.0; -9.089951014118482e-17 0.01726731646460112 0.0; 0.017267316464601216 0.01726731646460123 0.0; 0.04999999999999998 0.017267316464601185 0.0; 0.08273268353539881 0.017267316464601175 0.0; 0.09999999999999987 0.01726731646460112 0.0; -7.123931074678078e-18 0.0499999999999999 0.0; 0.017267316464601164 0.05 0.0; 0.04999999999999999 0.04999999999999993 0.0; 0.08273268353539884 0.04999999999999995 0.0; 0.09999999999999995 0.04999999999999996 0.0; -2.393918396847995e-16 0.0827326835353988 0.0; 0.017267316464601133 0.08273268353539881 0.0; 0.04999999999999998 0.08273268353539882 0.0; 0.08273268353539881 0.08273268353539874 0.0; 0.0

100-element Vector{Vector{Int64}}:
 [1, 2, 3, 4, 5, 6, 7, 8, 9, 10  …  16, 17, 18, 19, 20, 21, 22, 23, 24, 25]
 [21, 22, 23, 24, 25, 26, 27, 28, 29, 30  …  36, 37, 38, 39, 40, 41, 42, 43, 44, 45]
 [41, 42, 43, 44, 45, 46, 47, 48, 49, 50  …  56, 57, 58, 59, 60, 61, 62, 63, 64, 65]
 [61, 62, 63, 64, 65, 66, 67, 68, 69, 70  …  76, 77, 78, 79, 80, 81, 82, 83, 84, 85]
 [81, 82, 83, 84, 85, 86, 87, 88, 89, 90  …  96, 97, 98, 99, 100, 101, 102, 103, 104, 105]
 [101, 102, 103, 104, 105, 106, 107, 108, 109, 110  …  116, 117, 118, 119, 120, 121, 122, 123, 124, 125]
 [121, 122, 123, 124, 125, 126, 127, 128, 129, 130  …  136, 137, 138, 139, 140, 141, 142, 143, 144, 145]
 [141, 142, 143, 144, 145, 146, 147, 148, 149, 150  …  156, 157, 158, 159, 160, 161, 162, 163, 164, 165]
 [161, 162, 163, 164, 165, 166, 167, 168, 169, 170  …  176, 177, 178, 179, 180, 181, 182, 183, 184, 185]
 [181, 182, 183, 184, 185, 186, 187, 188, 189, 190  …  196, 197, 198, 199, 200, 201, 202, 203, 204, 205]
 [5, 206, 207, 208

In [61]:
sem.inverse_error

4.884981308350689e-15